# `gpu_hours.py` on Google Colab (GPU)

1. **Runtime → Change runtime type → GPU** (T4/L4/A100, etc.).
2. Upload this folder (and the Mamba interpretability repos) under `/content`, **or** zip them, upload, and unzip so the layout matches the paths in the next cell.
3. Run cells top to bottom.

**Required layout** (same as on your machine, but under `/content`):

- `/content/gpu_hours/` — this repo (`gpu_hours.py`, `requirements.txt`, …)
- `/content/mamba_interpretability_1/` — vanilla loader (`mamba_model_loader.py`)
- `/content/mamba_steered_interpretability_1/` — steered stack
- `/content/mamba_stable_interpretability_1/mamba_2_interpretability_1/` — Mamba2 attach stack

If you only upload `gpu_hours` and one repo, pass `--variants` accordingly (e.g. `--variants mamba` only).

In [ ]:
# GPU sanity check (should show a CUDA device)
!nvidia-smi

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Colab usually ships torch+CUDA; install the rest for HF + datasets.
%pip install -q "transformers>=4.40" "datasets>=2.14" accelerate tqdm sentencepiece protobuf

## Paths and run command

Edit `CONTENT` if you put everything somewhere else (e.g. Drive: `/content/drive/MyDrive/gpu_colab`).

In [ ]:
import os
import shlex
import subprocess
import sys

# --- edit if your upload root is not /content ---
CONTENT = "/content"

GPU_HOURS_DIR = os.path.join(CONTENT, "gpu_hours")
MAMBA_REPO = os.path.join(CONTENT, "mamba_interpretability_1")
STEERED_REPO = os.path.join(CONTENT, "mamba_steered_interpretability_1")
MAMBA2_REPO = os.path.join(CONTENT, "mamba_stable_interpretability_1", "mamba_2_interpretability_1")

SCRIPT = os.path.join(GPU_HOURS_DIR, "gpu_hours.py")
assert os.path.isfile(SCRIPT), f"Missing {SCRIPT} — upload gpu_hours or fix GPU_HOURS_DIR"

# All three stacks, prediction-only on GPU (add --workloads pred train for default training phase)
cmd = [
    sys.executable,
    SCRIPT,
    "--device",
    "cuda",
    "--mamba-repo",
    MAMBA_REPO,
    "--steered-repo",
    STEERED_REPO,
    "--mamba2-repo",
    MAMBA2_REPO,
    "--workloads",
    "pred",
    "--variants",
    "mamba",
    "steered",
    "mamba2",
    "--model",
    "state-spaces/mamba-130m-hf",
]

print("Run command:\n", " ".join(shlex.quote(c) for c in cmd), "\n", sep="")
proc = subprocess.run(cmd, cwd=GPU_HOURS_DIR)
raise SystemExit(proc.returncode)

## Shell equivalent (optional)

From a Colab terminal or `!` line, after `cd` into `gpu_hours`:

```bash
python gpu_hours.py --device cuda \
  --mamba-repo /content/mamba_interpretability_1 \
  --steered-repo /content/mamba_steered_interpretability_1 \
  --mamba2-repo /content/mamba_stable_interpretability_1/mamba_2_interpretability_1 \
  --workloads pred \
  --variants mamba steered mamba2 \
  --model state-spaces/mamba-130m-hf
```

Add HF auth if needed: `from huggingface_hub import login; login()` or set the `HF_TOKEN` secret in Colab.